In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import tensorflow as tf
import numpy as np

!unzip "/content/drive/MyDrive/moondata/moon_massive.zip" -d "/content/moond"

df = pd.read_csv("/content/moond/moon_massive/dataset.csv")

# prepend full path
df["image_path"] = "/content/moond/moon_massive/" + df["image_path"] + ".png"
df.head()

Streaming output truncated to the last 5000 lines.
  inflating: /content/moond/moon_massive/wxg/wxg_0924.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0706.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0968.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0404.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0800.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0986.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0850.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0820.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0281.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0099.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0927.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0786.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0486.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0506.png  
  inflating: /content/moond/moon_massive/wxg/wxg_0141.png  
  inflating: /content/moond/moon_massive/wxg/wxg_

,image_path,illumination,fov,limit_mag,phase,phase_angle,age,date,phase_name
0,/content/moond/moon_massive/wxc/wxc_0000.png,5.679802,0.58,3,0.056798,2.660314,2.25,2025-11-23T00:20:05,WXC
1,/content/moond/moon_massive/wxc/wxc_0001.png,6.080707,0.60,4,0.060807,2.643271,2.31,2025-11-23T01:50:07,WXC
2,/content/moond/moon_massive/wxc/wxc_0002.png,6.498431,1.00,5,0.064984,2.626062,2.36,2025-11-23T03:20:09,WXC
3,/content/moond/moon_massive/wxc/wxc_0003.png,6.913473,1.50,6,0.069135,2.609467,2.42,2025-11-23T04:50:11,WXC
4,/content/moond/moon_massive/wxc/wxc_0004.png,7.306273,2.00,7,0.073063,2.594181,2.47,2025-11-23T06:20:12,WXC


In [ ]:
from sklearn.model_selection import train_test_split

_, train_df_full = train_test_split(
    df, test_size=0.50, random_state=42, stratify=df["phase_name"]
)

train_df, temp_df = train_test_split(
    train_df_full, test_size=0.20, random_state=42, stratify=train_df_full["phase_name"]
)

test_df, val_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["phase_name"]
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import tensorflow as tf

train_aug = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    zoom_range=0.1,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=10.0,
    horizontal_flip=False,
    vertical_flip=False,
    brightness_range=[0.8, 1.15]
)

valid_aug = ImageDataGenerator(preprocessing_function=preprocess_input)
test_aug  = ImageDataGenerator(preprocessing_function=preprocess_input)


In [ ]:
target_size = (299, 299)

train_flow = train_aug.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="phase_name",
    target_size=target_size,
    class_mode="categorical",
    batch_size=32,
    shuffle=True
)

valid_flow = valid_aug.flow_from_dataframe(
    val_df,
    x_col="image_path",
    y_col="phase_name",
    target_size=target_size,
    class_mode="categorical",
    batch_size=32,
    shuffle=False
)

test_flow = test_aug.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="phase_name",
    target_size=target_size,
    class_mode="categorical",
    batch_size=32,
    shuffle=False
)


Found 3276 validated image filenames belonging to 8 classes.
Found 410 validated image filenames belonging to 8 classes.
Found 410 validated image filenames belonging to 8 classes.


In [ ]:
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers


def build_model(base, num_classes=8):
  L2 = regularizers.l2(1e-3)

  x = base.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(256, activation='relu', kernel_regularizer=L2)(x)
  x = Dropout(0.4)(x)
  x = Dense(128, activation='relu', kernel_regularizer=L2)(x)
  x = Dropout(0.4)(x)
  x = Dense(64, activation='relu', kernel_regularizer=L2)(x)
  x = Dropout(0.3)(x)
  out = Dense(num_classes, activation='softmax')(x)
  model = Model(inputs=base.input, outputs=out)
  return model


In [ ]:
base1 = tf.keras.applications.MobileNetV2(
    include_top=False, weights="imagenet",
    input_shape=(224,224,3)
)
for layer in base1.layers[:60]:
    layer.trainable = False

for layer in base1.layers[60:]:
    layer.trainable = True

model1 = build_model(base1)
model1.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])

print("Training MobileNetV2...")
model1.fit(train_flow, validation_data=valid_flow, epochs=20)


Training MobileNetV2...
Epoch 1/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 115s 814ms/step - accuracy: 0.3418 - loss: 2.3583 - val_accuracy: 0.1561 - val_loss: 10.6758
Epoch 2/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 60s 586ms/step - accuracy: 0.7304 - loss: 1.3761 - val_accuracy: 0.4585 - val_loss: 4.8677
Epoch 3/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 61s 595ms/step - accuracy: 0.8138 - loss: 1.0722 - val_accuracy: 0.1537 - val_loss: 8.9445
Epoch 4/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 61s 594ms/step - accuracy: 0.8777 - loss: 0.8355 - val_accuracy: 0.3512 - val_loss: 7.4103
Epoch 5/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 60s 584ms/step - accuracy: 0.8812 - loss: 0.7530 - val_accuracy: 0.1829 - val_loss: 15.8428
Epoch 6/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 69s 669ms/step - accuracy: 0.9141 - loss: 0.6464 - val_accuracy: 0.2220 - val_loss: 10.3812
Epoch 7/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 62s 602ms/step - accuracy: 0.9178 - loss: 0.5968 - val_accuracy: 0.2976 - val_loss: 6.7479
Epoch 8/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 63s 612ms/step 

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

# --------------------
# Basic Residual Block
# --------------------
def BasicBlock(x, filters, stride=1, use_projection=False, l2=1e-5):
    shortcut = x

    # First conv
    x = layers.Conv2D(filters, kernel_size=3, strides=stride, padding='same',
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second conv
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same',
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)

    # Projection for shape mismatch
    if use_projection:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=stride,
                                 kernel_initializer='he_normal',
                                 kernel_regularizer=regularizers.l2(l2))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Residual add
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


# --------------------
# Build ResNet-18
# --------------------
def build_resnet18(input_shape=(224,224,3), num_classes=3, l2=1e-5, dropout_rate=0.4):

    inputs = layers.Input(shape=input_shape)

    # Initial Conv 7×7, stride 2, padding 3
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same',
                      kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(l2))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # MaxPool 3×3 stride 2
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Residual Layer Groups: (2,2,2,2)
    # Group 1: filters=64
    x = BasicBlock(x, 64)
    x = BasicBlock(x, 64)

    # Group 2: filters=128
    x = BasicBlock(x, 128, stride=2, use_projection=True)
    x = BasicBlock(x, 128)

    # Group 3: filters=256
    x = BasicBlock(x, 256, stride=2, use_projection=True)
    x = BasicBlock(x, 256)

    # Group 4: filters=512
    x = BasicBlock(x, 512, stride=2, use_projection=True)
    x = BasicBlock(x, 512)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dropout for preventing overfitting
    x = layers.Dropout(dropout_rate)(x)

    # Final Softmax Layer
    outputs = layers.Dense(num_classes, activation='softmax',
                           kernel_initializer="he_normal")(x)

    model = models.Model(inputs, outputs)
    return model


# --------------------
# BUILD + COMPILE MODEL
# --------------------
num_classes = 8  # dynamic based on dataset

model_r18 = build_resnet18(
    input_shape=(224,224,3),
    num_classes=num_classes,
    l2=1e-5,
    dropout_rate=0.4
)

model_r18.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_r18.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 112, 112,  │      9,472 │ input_layer_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 112, 112,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 56, 56,    │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 56, 56,    │     36,928 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     36,928 │ re_lu_1[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 56, 56,    │          0 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 56, 56,    │     36,928 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 56, 56,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │     36,928 │ re_lu_3[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv2d_4[0][0]  

 Total params: 11,195,016 (42.71 MB)

 Trainable params: 11,185,416 (42.67 MB)

 Non-trainable params: 9,600 (37.50 KB)

In [ ]:
def build_xception(num_classes=8, dropout_rate=0.4, l2=1e-5):

    base = tf.keras.applications.Xception(
        include_top=False,
        weights="imagenet",
        input_shape=(299, 299, 3)
    )
    base.trainable = True

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation="relu",
                     kernel_initializer="he_normal",
                     kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(128, activation="relu",
                     kernel_initializer="he_normal",
                     kernel_regularizer=regularizers.l2(l2))(x)

    out = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base.input, outputs=out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


model_xcp = build_xception(num_classes=8)


In [ ]:
model_xcp.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer_1[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 74, 74,    │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add_12[0][0]    

 Total params: 21,977,264 (83.84 MB)

 Trainable params: 21,922,736 (83.63 MB)

 Non-trainable params: 54,528 (213.00 KB)

In [ ]:
history = model_r18.fit(
    train_flow,
    validation_data=valid_flow,
    epochs=25
)

Epoch 1/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 111s 798ms/step - accuracy: 0.3498 - loss: 2.1005 - val_accuracy: 0.2293 - val_loss: 11.1081
Epoch 2/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 63s 612ms/step - accuracy: 0.5492 - loss: 1.2508 - val_accuracy: 0.4415 - val_loss: 2.8339
Epoch 3/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 62s 603ms/step - accuracy: 0.6362 - loss: 1.0208 - val_accuracy: 0.6878 - val_loss: 1.0241
Epoch 4/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 63s 606ms/step - accuracy: 0.6870 - loss: 0.9086 - val_accuracy: 0.7854 - val_loss: 0.6588
Epoch 5/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 63s 608ms/step - accuracy: 0.7348 - loss: 0.8223 - val_accuracy: 0.2683 - val_loss: 4.0265
Epoch 6/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 62s 601ms/step - accuracy: 0.7592 - loss: 0.7521 - val_accuracy: 0.4000 - val_loss: 2.2760
Epoch 7/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 62s 598ms/step - accuracy: 0.8163 - loss: 0.6138 - val_accuracy: 0.5049 - val_loss: 2.2495
Epoch 8/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 63s 614ms/step - accuracy: 0.8015 - loss:

In [ ]:
history_xcp = model_xcp.fit(
    train_flow,
    validation_data=valid_flow,
    epochs=25
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 331s 2s/step - accuracy: 0.3050 - loss: 1.8144 - val_accuracy: 0.3683 - val_loss: 5.8394
Epoch 2/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.7925 - loss: 0.6826 - val_accuracy: 0.2902 - val_loss: 9.4887
Epoch 3/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - accuracy: 0.8575 - loss: 0.4237 - val_accuracy: 0.8512 - val_loss: 0.4220
Epoch 4/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.8898 - loss: 0.3469 - val_accuracy: 0.7585 - val_loss: 1.0273
Epoch 5/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9272 - loss: 0.2410 - val_accuracy: 0.8634 - val_loss: 0.6021
Epoch 6/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9320 - loss: 0.2320 - val_accuracy: 0.9317 - val_loss: 0.2414
Epoch 7/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9483 - loss: 0.1811 - val_accuracy: 0.8976 - val_loss: 0.3511
Epoch 8/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9295 - loss: 0.2135 - val_accu

In [ ]:
import matplotlib.pyplot as plt

acc = history_xcp.history['accuracy']
val_acc = history_xcp.history['val_accuracy']
loss = history_xcp.history['loss']
val_loss = history_xcp.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Train Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy (Xception)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss (Xception)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


NameError: name 'history_xcp' is not defined

In [ ]:
base2 = tf.keras.applications.ResNet50(
    include_top=False, weights="imagenet",
    input_shape=(256,256,3)
)
base2.trainable = False

model2 = build_model(base2)
model2.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("Training ResNet50...")
model2.fit(train_flow, validation_data=valid_flow, epochs=10)


In [ ]:
base3 = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(256,256,3)
)
base3.trainable = False

model3 = build_model(base3)
model3.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("Training EfficientNetB0...")
model3.fit(train_flow, validation_data=valid_flow, epochs=10)


In [ ]:
#print("MobileNetV2 Test:")
#print(model1.evaluate(test_flow))

print("ResNet18 Test:")
print(model_r18.evaluate(test_flow))

print("XceptionNet Test:")
print(model_xcp.evaluate(test_flow))

#print("EfficientNetB0 Test:")
#print(model3.evaluate(test_flow))


ResNet18 Test:


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 555ms/step - accuracy: 0.4652 - loss: 2.2822
[2.328179121017456, 0.4853658676147461]
XceptionNet Test:
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 313ms/step - accuracy: 0.9739 - loss: 0.1323
[0.16357199847698212, 0.9634146094322205]
